In [1]:
import pandas as pd
import numpy as np

import xgboost as xgb
from math import exp

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, mean_absolute_error

pd.set_option('display.max_columns',50)

In [2]:
teams = pd.read_csv('./data/MTeams.csv')
print("Teams data shape -->", teams.shape)
seasons = pd.read_csv('./data/MSeasons.csv')
print("Seasons data shape -->", seasons.shape)
tourney_seeds = pd.read_csv('./data/MNCAATourneySeeds.csv')
print("Tourney Seeds data shape -->", tourney_seeds.shape)
tourney_results = pd.read_csv('./data/MNCAATourneyDetailedResults.csv')
print("Tourney Results shape -->", tourney_results.shape)
detailed_regular = pd.read_csv('./data/MRegularSeasonDetailedResults.csv')
print("Detailed Regular data shape -->", detailed_regular.shape)
massey_ordinals = pd.read_csv('./data/MMasseyOrdinals.csv')
print("Massey Ordinals data shape -->", massey_ordinals.shape)

Teams data shape --> (380, 4)
Seasons data shape --> (41, 6)
Tourney Seeds data shape --> (2558, 3)
Tourney Results shape --> (1382, 34)
Detailed Regular data shape --> (118449, 34)
Massey Ordinals data shape --> (5489117, 5)


In [3]:
massey_ordinals

,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
0,2003,35,SEL,1102,159
1,2003,35,SEL,1103,229
2,2003,35,SEL,1104,12
3,2003,35,SEL,1105,314
4,2003,35,SEL,1106,260
...,...,...,...,...,...
5489112,2025,107,WOL,1476,282
5489113,2025,107,WOL,1477,346
5489114,2025,107,WOL,1478,325
5489115,2025,107,WOL,1479,306


In [4]:
massey_ordinals['SystemName'].value_counts()

SystemName
MOR    141719
POM    139228
SAG    130371
DOK    125932
MAS    115626
        ...  
CRW       351
HRN       351
PMC       351
BP5       345
PH        326
Name: count, Length: 192, dtype: int64

In [5]:
detailed_regular.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='object')

In [6]:
tourney_results.tail()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
1377,2024,146,1301,76,1181,64,N,0,28,60,3,13,17,23,8,27,16,4,4,6,16,19,59,5,20,21,26,10,27,11,9,4,5,23
1378,2024,146,1345,72,1397,66,N,0,24,53,3,15,21,33,8,32,16,10,5,2,12,24,62,11,26,7,11,6,17,17,6,8,4,25
1379,2024,152,1163,86,1104,72,N,0,31,62,10,25,14,18,10,25,20,4,4,8,17,26,58,11,23,9,11,7,21,9,7,2,5,15
1380,2024,152,1345,63,1301,50,N,0,22,55,10,25,9,10,10,28,13,14,5,2,8,21,57,5,19,3,4,6,22,10,11,8,3,13
1381,2024,154,1163,75,1345,60,N,0,30,62,6,22,9,11,13,19,18,6,3,4,18,24,54,1,7,11,15,8,19,8,9,3,3,15


In [7]:
detailed_regular.tail()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
118444,2025,120,1433,71,1182,62,A,0,23,55,8,22,17,21,12,22,10,14,9,4,23,21,51,4,16,16,24,11,19,9,15,8,5,16
118445,2025,120,1436,79,1107,71,H,0,28,48,11,22,12,15,4,23,14,12,2,9,11,25,61,9,20,12,15,10,18,11,8,9,0,15
118446,2025,120,1438,60,1199,57,H,0,21,53,11,24,7,9,6,20,15,11,3,7,12,23,62,7,22,4,7,12,21,12,8,7,9,15
118447,2025,120,1452,71,1428,69,A,0,26,57,8,23,11,16,6,21,14,10,7,2,20,19,50,9,28,22,32,10,26,11,16,7,3,17
118448,2025,120,1460,98,1237,85,H,0,36,57,14,30,12,15,10,24,27,11,2,4,11,31,58,14,24,9,14,4,12,16,4,7,1,16


In [8]:
detailed_regular['WEffFGPerc'] = 100 * (detailed_regular['WFGM'] + (0.5 * detailed_regular['WFGM3']))/detailed_regular['WFGA']
detailed_regular['LEffFGPerc'] = 100 * (detailed_regular['LFGM'] + (0.5 * detailed_regular['LFGM3']))/detailed_regular['LFGA']
detailed_regular['WFTPerc'] = 100 * detailed_regular['WFTM']/(detailed_regular['WFTA']+1)
detailed_regular['LFTPerc'] = 100 * detailed_regular['LFTM']/(detailed_regular['LFTA']+1)
detailed_regular['WTOPerc'] = 100 * detailed_regular['WTO'] / (detailed_regular['WFGA'] - detailed_regular['WOR'] + detailed_regular['WTO'] + (0.44 * detailed_regular['WFTA']))
detailed_regular['LTOPerc'] = 100 * detailed_regular['LTO'] / (detailed_regular['LFGA'] - detailed_regular['LOR'] + detailed_regular['LTO'] + (0.44 * detailed_regular['LFTA']))
detailed_regular['WORPerc'] = 100 * detailed_regular['WOR'] / (detailed_regular['WOR'] + detailed_regular['LDR'])
detailed_regular['LORPerc'] = 100 * detailed_regular['LOR'] / (detailed_regular['LOR'] + detailed_regular['WDR'])
detailed_regular['WDefEff'] = (detailed_regular['WStl'] + detailed_regular['WBlk'])/ detailed_regular['WPF']
detailed_regular['LDefEff'] = (detailed_regular['LStl'] + detailed_regular['LBlk'])/ detailed_regular['LPF']

In [9]:
drop_cols = ['WLoc','NumOT','WFGM','WFGA','LFGM','LFGA','WFGM3','WFGA3','WStl','WBlk','WPF','LFGM3','LFGA3','WFTM','WFTA','LStl','LBlk','LPF','LFTM','LFTA','WOR','WDR','LOR','LDR','WTO','LTO']
detailed_regular.drop(drop_cols, inplace=True, axis=1)
detailed_regular

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WAst,LAst,WEffFGPerc,LEffFGPerc,WFTPerc,LFTPerc,WTOPerc,LTOPerc,WORPerc,LORPerc,WDefEff,LDefEff
0,2003,10,1104,68,1328,62,13,8,49.137931,43.396226,57.894737,69.565217,30.699413,25.466893,38.888889,29.411765,0.363636,0.550000
1,2003,10,1272,70,1393,63,16,7,48.387097,40.298507,50.000000,42.857143,19.016969,17.699115,37.500000,41.666667,0.444444,0.875000
2,2003,11,1266,73,1437,61,15,9,48.275862,32.191781,56.666667,58.333333,15.683814,18.714910,43.589744,54.385965,0.280000,0.304348
3,2003,11,1296,56,1457,50,11,9,51.315789,42.857143,53.125000,50.000000,20.818876,32.986111,23.076923,47.222222,0.888889,0.304348
4,2003,11,1400,77,1208,71,12,12,54.098361,43.548387,78.571429,60.714286,21.971124,15.903308,53.125000,48.837209,0.400000,0.571429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118444,2025,120,1433,71,1182,62,10,9,49.090909,45.098039,77.272727,64.000000,21.135266,22.879805,38.709677,33.333333,0.565217,0.812500
118445,2025,120,1436,79,1107,71,14,11,69.791667,48.360656,75.000000,75.000000,19.169329,12.195122,18.181818,30.303030,1.000000,0.600000
118446,2025,120,1438,60,1199,57,15,12,50.000000,42.741935,70.000000,50.000000,17.753389,13.097577,22.222222,37.500000,0.833333,1.066667
118447,2025,120,1452,71,1428,69,14,11,52.631579,47.000000,64.705882,66.666667,14.697237,22.831050,18.750000,32.258065,0.450000,0.588235


In [10]:
detailed_regular.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WAst',
       'LAst', 'WEffFGPerc', 'LEffFGPerc', 'WFTPerc', 'LFTPerc', 'WTOPerc',
       'LTOPerc', 'WORPerc', 'LORPerc', 'WDefEff', 'LDefEff'],
      dtype='object')

In [11]:
def assign_teams(df):
    # true means winner goes to Team A
    coin = np.random.rand(len(df)) < 0.5

    # team A stats based on coin toss
    teamA = pd.DataFrame({
        'Season': df['Season'],
        'DayNum': df['DayNum'],
        'TeamID' : np.where(coin, df['WTeamID'], df['LTeamID']),
        'Score' :  np.where(coin, df['WScore'], df['LScore']),
        'ORPerc': np.where(coin, df['WORPerc'], df['LORPerc']),
        'Ast': np.where(coin, df['WAst'], df['LAst']),
        'TOPerc': np.where(coin, df['WTOPerc'], df['LTOPerc']),
        'DefEff': np.where(coin, df['WDefEff'], df['LDefEff']),
        'EffFG': np.where(coin, df['WEffFGPerc'], df['LEffFGPerc']),
        'FTPerc': np.where(coin, df['WFTPerc'], df['LFTPerc']),
    })

    # team B stats based on coin toss
    teamB = pd.DataFrame({
        'TeamID' : np.where(coin, df['LTeamID'], df['WTeamID']),
        'Score' :  np.where(coin, df['LScore'], df['WScore']),
        'ORPerc': np.where(coin, df['LORPerc'], df['WORPerc']),
        'Ast': np.where(coin, df['LAst'], df['WAst']),
        'TOPerc': np.where(coin, df['LTOPerc'], df['WTOPerc']),
        'DefEff': np.where(coin, df['LDefEff'], df['WDefEff']),
        'EffFG': np.where(coin, df['LEffFGPerc'], df['WEffFGPerc']),
        'FTPerc': np.where(coin, df['LFTPerc'], df['WFTPerc']),
    })

    # Create Win column: 1 if Team A gets the winner's stats, 0 otherwise
    win = pd.Series(np.where(coin, 1, 0), name='Win')

    # Optionally, rename columns to clearly indicate Team A and B stats (except for Season)
    teamA = teamA.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_A')
    teamB = teamB.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_B')

    # Concatenate the two teams' stats and the Win column into one DataFrame
    new_df = pd.concat([teamA, teamB, win], axis=1)
    
    return new_df

new_detailed_reg = assign_teams(detailed_regular)
new_detailed_reg.head()

,Season,DayNum,TeamID_A,Score_A,ORPerc_A,Ast_A,TOPerc_A,DefEff_A,EffFG_A,FTPerc_A,TeamID_B,Score_B,ORPerc_B,Ast_B,TOPerc_B,DefEff_B,EffFG_B,FTPerc_B,Win
0,2003,10,1104,68,38.888889,13,30.699413,0.363636,49.137931,57.894737,1328,62,29.411765,8,25.466893,0.550000,43.396226,69.565217,1
1,2003,10,1393,63,41.666667,7,17.699115,0.875000,40.298507,42.857143,1272,70,37.500000,16,19.016969,0.444444,48.387097,50.000000,0
2,2003,11,1266,73,43.589744,15,15.683814,0.280000,48.275862,56.666667,1437,61,54.385965,9,18.714910,0.304348,32.191781,58.333333,1
3,2003,11,1457,50,47.222222,9,32.986111,0.304348,42.857143,50.000000,1296,56,23.076923,11,20.818876,0.888889,51.315789,53.125000,0
4,2003,11,1208,71,48.837209,12,15.903308,0.571429,43.548387,60.714286,1400,77,53.125000,12,21.971124,0.400000,54.098361,78.571429,0


In [12]:
new_detailed_reg['Win'].value_counts()

Win
0    59398
1    59051
Name: count, dtype: int64

In [13]:
new_detailed_reg.columns

Index(['Season', 'DayNum', 'TeamID_A', 'Score_A', 'ORPerc_A', 'Ast_A',
       'TOPerc_A', 'DefEff_A', 'EffFG_A', 'FTPerc_A', 'TeamID_B', 'Score_B',
       'ORPerc_B', 'Ast_B', 'TOPerc_B', 'DefEff_B', 'EffFG_B', 'FTPerc_B',
       'Win'],
      dtype='object')

In [14]:
df = new_detailed_reg.copy()

# -------------------------------------------
# STEP 1: Compute Differential Columns for Each Matchup
# -------------------------------------------
# For Team A perspective:
df['Score_diff_A']   = df['Score_A']   - df['Score_B']
df['ORPerc_diff_A']  = df['ORPerc_A']  - df['ORPerc_B']
df['Ast_diff_A']     = df['Ast_A']     - df['Ast_B']
df['TOPerc_diff_A']  = df['TOPerc_A']  - df['TOPerc_B']
df['DefEff_diff_A']  = df['DefEff_A']  - df['DefEff_B']
df['EffFG_diff_A']   = df['EffFG_A']   - df['EffFG_B']
df['FTPerc_diff_A']  = df['FTPerc_A']  - df['FTPerc_B']

# For Team B perspective (reverse the differential):
df['Score_diff_B']   = df['Score_B']   - df['Score_A']
df['ORPerc_diff_B']  = df['ORPerc_B']  - df['ORPerc_A']
df['Ast_diff_B']     = df['Ast_B']     - df['Ast_A']
df['TOPerc_diff_B']  = df['TOPerc_B']  - df['TOPerc_A']
df['DefEff_diff_B']  = df['DefEff_B']  - df['DefEff_A']
df['EffFG_diff_B']   = df['EffFG_B']   - df['EffFG_A']
df['FTPerc_diff_B']  = df['FTPerc_B']  - df['FTPerc_A']

# Create the regression target: Point Margin = Score_A - Score_B.
df['Point_Margin'] = df['Score_A'] - df['Score_B']

# -------------------------------------------
# STEP 2: Convert Wide Data to Long Format
# -------------------------------------------
# For rolling computations it is easier if each row represents a single team's performance.

# Create a DataFrame for Team A rows:
df_A = df[['Season', 'DayNum', 'TeamID_A', 
           'Score_diff_A', 'ORPerc_diff_A', 'Ast_diff_A', 'TOPerc_diff_A',
           'DefEff_diff_A', 'EffFG_diff_A', 'FTPerc_diff_A']].copy()
df_A.rename(columns={
    'TeamID_A': 'Team',
    'Score_diff_A': 'Score_diff',
    'ORPerc_diff_A': 'ORPerc_diff',
    'Ast_diff_A': 'Ast_diff',
    'TOPerc_diff_A': 'TOPerc_diff',
    'DefEff_diff_A': 'DefEff_diff',
    'EffFG_diff_A': 'EffFG_diff',
    'FTPerc_diff_A': 'FTPerc_diff'
}, inplace=True)

# Create a DataFrame for Team B rows:
df_B = df[['Season', 'DayNum', 'TeamID_B', 
           'Score_diff_B', 'ORPerc_diff_B', 'Ast_diff_B', 'TOPerc_diff_B',
           'DefEff_diff_B', 'EffFG_diff_B', 'FTPerc_diff_B']].copy()
df_B.rename(columns={
    'TeamID_B': 'Team',
    'Score_diff_B': 'Score_diff',
    'ORPerc_diff_B': 'ORPerc_diff',
    'Ast_diff_B': 'Ast_diff',
    'TOPerc_diff_B': 'TOPerc_diff',
    'DefEff_diff_B': 'DefEff_diff',
    'EffFG_diff_B': 'EffFG_diff',
    'FTPerc_diff_B': 'FTPerc_diff'
}, inplace=True)

# Combine Team A and Team B rows into one long DataFrame:
df_long = pd.concat([df_A, df_B], ignore_index=True)

# -------------------------------------------
# STEP 3: Compute 7-Game Rolling Averages for Differential Stats
# -------------------------------------------
# Sort by Season, Team, and DayNum so that each team's games are in chronological order.
df_long.sort_values(by=['Season', 'Team', 'DayNum'], inplace=True)

# Define a function to compute rolling averages using a 7-game window.
def compute_rolling(group, window=7):
    group = group.copy()
    group['Score_diff_7']  = group['Score_diff'].rolling(window=window, min_periods=window).mean()
    group['ORPerc_diff_7'] = group['ORPerc_diff'].rolling(window=window, min_periods=window).mean()
    group['Ast_diff_7']    = group['Ast_diff'].rolling(window=window, min_periods=window).mean()
    group['TOPerc_diff_7'] = group['TOPerc_diff'].rolling(window=window, min_periods=window).mean()
    group['DefEff_diff_7'] = group['DefEff_diff'].rolling(window=window, min_periods=window).mean()
    group['EffFG_diff_7']  = group['EffFG_diff'].rolling(window=window, min_periods=window).mean()
    group['FTPerc_diff_7'] = group['FTPerc_diff'].rolling(window=window, min_periods=window).mean()
    return group

df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)
df_long.reset_index(drop=True, inplace=True)

# -------------------------------------------
# STEP 4: Merge Rolling Stats Back into the Original (Wide) Matchup-Level DataFrame
# -------------------------------------------
# We need to merge the rolling stats for Team A and Team B back into the original DataFrame.

# Specify the columns from the long DataFrame that contain the rolling averages:
rolling_cols = ['Season', 'DayNum', 'Team', 
                'Score_diff_7', 'ORPerc_diff_7', 'Ast_diff_7', 'TOPerc_diff_7',
                'DefEff_diff_7', 'EffFG_diff_7', 'FTPerc_diff_7']

# Extract rolling stats for Team A:
rolling_A = df_long[rolling_cols].copy()
df = df.merge(rolling_A, left_on=['Season', 'DayNum', 'TeamID_A'],
              right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_A'))
df.rename(columns={
    'Score_diff_7': 'Score_diff_7_A',
    'ORPerc_diff_7': 'ORPerc_diff_7_A',
    'Ast_diff_7': 'Ast_diff_7_A',
    'TOPerc_diff_7': 'TOPerc_diff_7_A',
    'DefEff_diff_7': 'DefEff_diff_7_A',
    'EffFG_diff_7': 'EffFG_diff_7_A',
    'FTPerc_diff_7': 'FTPerc_diff_7_A'
}, inplace=True)
df.drop('Team', axis=1, inplace=True)

# Extract rolling stats for Team B:
rolling_B = df_long[rolling_cols].copy()
df = df.merge(rolling_B, left_on=['Season', 'DayNum', 'TeamID_B'],
              right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_B'))
df.rename(columns={
    'Score_diff_7': 'Score_diff_7_B',
    'ORPerc_diff_7': 'ORPerc_diff_7_B',
    'Ast_diff_7': 'Ast_diff_7_B',
    'TOPerc_diff_7': 'TOPerc_diff_7_B',
    'DefEff_diff_7': 'DefEff_diff_7_B',
    'EffFG_diff_7': 'EffFG_diff_7_B',
    'FTPerc_diff_7': 'FTPerc_diff_7_B'
}, inplace=True)
df.drop('Team', axis=1, inplace=True)

# -------------------------------------------
# STEP 5: Create Final DataFrame and Drop Original Columns
# -------------------------------------------
# We keep only the keys, the rolling stats, and target variables.
final_cols = ['Season', 'DayNum', 'TeamID_A', 'TeamID_B',
              'ORPerc_diff_7_A', 'TOPerc_diff_7_A',
              'DefEff_diff_7_A', 'EffFG_diff_7_A', 'FTPerc_diff_7_A',
              'ORPerc_diff_7_B', 'TOPerc_diff_7_B',
              'DefEff_diff_7_B', 'EffFG_diff_7_B', 'FTPerc_diff_7_B',
              'Win', 'Point_Margin']

df_final = df[final_cols].dropna().reset_index(drop=True)

/var/folders/hf/pd1tc6yd0yl0_l01c6nlywhc0000gn/T/ipykernel_73392/1364138302.py:83: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)


In [15]:
df_final

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
0,2003,33,1232,1183,-0.951985,-2.758769,0.120962,-4.983341,-12.428746,-2.439401,-2.613924,-0.132108,-7.448693,7.649894,1,6
1,2003,33,1292,1237,0.975224,5.027691,-0.311933,-0.551846,-6.643701,0.430207,2.448383,-0.130675,-15.615107,0.690647,1,4
2,2003,35,1231,1435,0.979515,-0.298213,0.387308,8.557617,11.411400,0.485802,-1.763459,-0.014092,13.656050,-6.894016,1,17
3,2003,35,1250,1162,1.180624,3.828888,0.077071,1.456485,11.409258,-6.304264,1.474739,-0.283262,-11.607017,-10.361916,1,16
4,2003,37,1140,1360,0.619789,1.986108,0.035920,13.024963,10.488572,10.422564,6.703842,-0.103381,0.635117,-5.081380,1,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91921,2025,120,1433,1182,14.642969,0.508599,0.058398,10.891320,11.473859,5.972062,-0.972135,0.093358,2.658062,-6.736436,1,9
91922,2025,120,1436,1107,2.660237,0.481976,0.248424,10.559537,2.689899,9.639510,-4.139265,0.084021,-3.431152,-8.910943,1,8
91923,2025,120,1438,1199,-9.325078,-0.224596,-0.090043,0.585391,5.455621,-4.874739,1.395151,0.091202,-3.692508,-9.566110,1,3
91924,2025,120,1428,1452,9.576996,3.076430,-0.220960,-0.816178,-8.297633,-3.016880,-1.584992,0.026739,-1.002182,0.564173,0,-2


In [16]:
df_final.corr()

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
Season,1.000000,-0.061898,0.018759,0.015049,0.000734,0.003707,0.002450,0.003176,0.000433,0.001560,-0.001942,-0.003939,-0.003580,-0.002831,-0.003300,-0.002416
DayNum,-0.061898,1.000000,-0.000087,0.003613,0.007943,-0.007461,0.006866,0.015916,0.004888,0.000622,-0.004979,0.004520,0.010082,0.003734,0.000777,-0.000966
TeamID_A,0.018759,-0.000087,1.000000,-0.000124,0.041015,-0.015209,0.021489,0.028403,0.016437,0.003645,-0.002303,0.007167,0.005307,0.000213,0.021034,0.021411
TeamID_B,0.015049,0.003613,-0.000124,1.000000,0.002765,-0.003058,0.011069,0.007209,0.006007,0.035266,-0.017173,0.022838,0.028902,0.014609,-0.025528,-0.021893
ORPerc_diff_7_A,0.000734,0.007943,0.041015,0.002765,1.000000,0.096106,0.144118,0.214537,0.073603,-0.176397,-0.038823,0.012074,-0.008022,0.007307,0.189380,0.254140
TOPerc_diff_7_A,0.003707,-0.007461,-0.015209,-0.003058,0.096106,1.000000,-0.531674,-0.072334,-0.080307,-0.039281,-0.185307,0.072863,-0.016814,0.006559,-0.171743,-0.222239
DefEff_diff_7_A,0.002450,0.006866,0.021489,0.011069,0.144118,-0.531674,1.000000,0.443058,0.190904,0.015538,0.076324,-0.124337,-0.024599,-0.013962,0.284470,0.331455
EffFG_diff_7_A,0.003176,0.015916,0.028403,0.007209,0.214537,-0.072334,0.443058,1.000000,0.175697,-0.002416,-0.012352,-0.024268,-0.127602,-0.011777,0.368854,0.458458
FTPerc_diff_7_A,0.000433,0.004888,0.016437,0.006007,0.073603,-0.080307,0.190904,0.175697,1.000000,0.003771,0.010943,-0.013792,-0.010941,-0.172257,0.152890,0.171605
ORPerc_diff_7_B,0.001560,0.000622,0.003645,0.035266,-0.176397,-0.039281,0.015538,-0.002416,0.003771,1.000000,0.091042,0.142156,0.213426,0.073352,-0.188458,-0.248111


In [17]:
df_final['Season'].value_counts()

Season
2024    4437
2023    4420
2019    4310
2018    4254
2017    4245
2016    4206
2014    4188
2015    4184
2020    4173
2013    4171
2022    4169
2011    4106
2010    4103
2012    4103
2009    4089
2025    4035
2008    4010
2007    3915
2006    3625
2005    3560
2003    3518
2004    3476
2021    2629
Name: count, dtype: int64

In [18]:
train_data = df_final[df_final['Season'].isin(range(2003,2022))]
val_data = df_final[df_final['Season'].isin(range(2022,2025))].reset_index(drop=True)

In [19]:
train_data

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
0,2003,33,1232,1183,-0.951985,-2.758769,0.120962,-4.983341,-12.428746,-2.439401,-2.613924,-0.132108,-7.448693,7.649894,1,6
1,2003,33,1292,1237,0.975224,5.027691,-0.311933,-0.551846,-6.643701,0.430207,2.448383,-0.130675,-15.615107,0.690647,1,4
2,2003,35,1231,1435,0.979515,-0.298213,0.387308,8.557617,11.411400,0.485802,-1.763459,-0.014092,13.656050,-6.894016,1,17
3,2003,35,1250,1162,1.180624,3.828888,0.077071,1.456485,11.409258,-6.304264,1.474739,-0.283262,-11.607017,-10.361916,1,16
4,2003,37,1140,1360,0.619789,1.986108,0.035920,13.024963,10.488572,10.422564,6.703842,-0.103381,0.635117,-5.081380,1,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74860,2021,132,1261,1104,-0.241140,-0.480218,-0.163426,2.042670,5.814195,3.818984,-2.908723,0.077119,3.400263,-0.205810,0,-1
74861,2021,132,1159,1259,3.229653,-1.588250,0.347862,10.242506,-1.102819,5.532620,4.356403,-0.155316,2.172787,-8.972143,1,13
74862,2021,132,1222,1153,15.748633,-6.082546,0.278850,12.666525,19.226137,-0.874018,1.007237,0.006553,0.051513,-3.549972,1,37
74863,2021,132,1326,1228,-3.551830,2.038594,-0.044528,1.156094,-0.035110,15.854555,2.969580,-0.030203,8.007888,0.568926,0,-3


In [20]:
train_data.columns

Index(['Season', 'DayNum', 'TeamID_A', 'TeamID_B', 'ORPerc_diff_7_A',
       'TOPerc_diff_7_A', 'DefEff_diff_7_A', 'EffFG_diff_7_A',
       'FTPerc_diff_7_A', 'ORPerc_diff_7_B', 'TOPerc_diff_7_B',
       'DefEff_diff_7_B', 'EffFG_diff_7_B', 'FTPerc_diff_7_B', 'Win',
       'Point_Margin'],
      dtype='object')

In [28]:
train_data.to_csv('./data/train_data.csv', index=False)
val_data.to_csv('./data/val_data.csv', index=False)

In [21]:
feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Point_Margin'  # We want to predict the margin

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_val = val_data[feature_cols]
y_val = val_data[target_col]

# For Brier score, we need the actual outcome: 1 if Team A won, 0 otherwise
y_val_binary = val_data['Win'].values

# -------------------------------------------------
# 2) FIT A BASELINE LINEAR REGRESSION MODEL
# -------------------------------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)

# -------------------------------------------------
# 3) PREDICT MARGINS ON VALIDATION SET
# -------------------------------------------------
pred_margin_val = lr.predict(X_val)

# -------------------------------------------------
# 4) CONVERT MARGINS -> PROBABILITIES
# -------------------------------------------------
def margin_to_probability(margin, scale=10.0):
    """
    Simple logistic transform from margin -> P(A wins).
    P = 1 / (1 + exp(-margin / scale))
    """
    return 1.0 / (1.0 + np.exp(-margin / scale))

pred_prob_val = margin_to_probability(pred_margin_val, scale=10.0)

# -------------------------------------------------
# 5) COMPUTE BRIER SCORE
# -------------------------------------------------
def brier_score(prob, outcome):
    return np.mean((prob - outcome)**2)

val_brier = brier_score(pred_prob_val, y_val_binary)
print("Baseline Linear Regression - Validation Brier Score:", val_brier)

Baseline Linear Regression - Validation Brier Score: 0.1728053581801779


In [22]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import norm

feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Point_Margin'

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_val = val_data[feature_cols]
y_val = val_data[target_col]

val_data['actual_win'] = (val_data['Point_Margin'] > 0).astype(int)

'''param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1, 2],     # L1 regularization
    'reg_lambda': [1, 1.5, 2, 5]    # L2 regularization
}

# Initialize the base model
xgb_base = XGBRegressor(random_state=42)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_distributions,
    n_iter=50,               # number of random parameter sets to try
    scoring='neg_mean_absolute_error',      # using MAE as the objective
    cv=5,                    # 3-fold CV on your training data
    verbose=1,
    random_state=123,
    n_jobs=-1               # use all available cores
)

random_search.fit(X_train, y_train)

print("Best Params:", random_search.best_params_)
print("Best CV Score (neg MAE):", random_search.best_score_)

# Retrieve the best model
best_model = random_search.best_estimator_'''

'param_distributions = {\n    \'n_estimators\': [100, 200, 300, 500],\n    \'learning_rate\': [0.01, 0.05, 0.1, 0.2],\n    \'max_depth\': [3, 4, 5, 6],\n    \'subsample\': [0.6, 0.8, 1.0],\n    \'colsample_bytree\': [0.6, 0.8, 1.0],\n    \'reg_alpha\': [0, 0.1, 1, 2],     # L1 regularization\n    \'reg_lambda\': [1, 1.5, 2, 5]    # L2 regularization\n}\n\n# Initialize the base model\nxgb_base = XGBRegressor(random_state=42)\n\n# RandomizedSearchCV\nrandom_search = RandomizedSearchCV(\n    estimator=xgb_base,\n    param_distributions=param_distributions,\n    n_iter=50,               # number of random parameter sets to try\n    scoring=\'neg_mean_absolute_error\',      # using MAE as the objective\n    cv=5,                    # 3-fold CV on your training data\n    verbose=1,\n    random_state=123,\n    n_jobs=-1               # use all available cores\n)\n\nrandom_search.fit(X_train, y_train)\n\nprint("Best Params:", random_search.best_params_)\nprint("Best CV Score (neg MAE):", rando

In [23]:
best_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.6,
    colsample_bytree=0.6,
    reg_lambda = 2,
    reg_alpha = 1,
    random_state = 123
)

best_model.fit(X_train, y_train)

val_pred_margin = best_model.predict(X_val)

# Convert margin to probability
def margin_to_probability(margin_array, scale=10.0):
    """
    Convert predicted margins to a win probability for Team A
    using a Normal CDF approach:
        P(A wins) = Phi((margin) / scale)
    """
    return norm.cdf(margin_array / scale)

val_pred_prob = margin_to_probability(val_pred_margin, scale=10.0)

val_data['pred_prob_A_wins'] = val_pred_prob

# Brier score function (assuming you already have it defined)
def brier_score(y_true, y_prob):
    return np.mean((y_true - y_prob)**2)

val_brier = brier_score(val_data['actual_win'], val_data['pred_prob_A_wins'])

print("Validation Brier Score with tuned model:", val_brier)

Validation Brier Score with tuned model: 0.16485098071019585


In [27]:
data_2025 = df_final[df_final['Season']==2025]
data_2025.tail()

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
91921,2025,120,1433,1182,14.642969,0.508599,0.058398,10.891320,11.473859,5.972062,-0.972135,0.093358,2.658062,-6.736436,1,9
91922,2025,120,1436,1107,2.660237,0.481976,0.248424,10.559537,2.689899,9.639510,-4.139265,0.084021,-3.431152,-8.910943,1,8
91923,2025,120,1438,1199,-9.325078,-0.224596,-0.090043,0.585391,5.455621,-4.874739,1.395151,0.091202,-3.692508,-9.566110,1,3
91924,2025,120,1428,1452,9.576996,3.076430,-0.220960,-0.816178,-8.297633,-3.016880,-1.584992,0.026739,-1.002182,0.564173,0,-2
91925,2025,120,1460,1237,2.588005,3.603584,0.037034,5.711587,-1.855842,-8.132976,-4.752507,0.023982,-10.174218,4.606278,1,13
